# Robustness: Tau Threshold (TCRI)

比較 TCRI 門檻 tau = 6 / 7 / 8 時，Logistic baseline (有 GICS, Platt 校準) 在測試集的表現。


In [1]:
!pip install -U pandas numpy scikit-learn shap matplotlib openpyxl



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, pandas as pd
TCRI_PATH = 'tcri.csv'
FIN_PATH  = 'ratios_filled_with_gics_category.csv'
ROOT_OUT  = 'outputs/robustness_tau'
os.makedirs(ROOT_OUT, exist_ok=True)
MERGED_PATH = os.path.join(ROOT_OUT, 'merged.csv')

!python merge_tcri_and_ratios.py \
  --tcri "$TCRI_PATH" \
  --ratios  "$FIN_PATH" \
  --ratios-date-format "%Y/%m" \
  --dedup-ratios \
  --out "$MERGED_PATH"

merged = pd.read_csv(MERGED_PATH, parse_dates=['mdate'])
merged.head()


[INFO] Dropped 88 duplicate rows from ratios data based on (id,date)
[INFO] Merged shape: (6383, 19). Saved to /Users/chieh.1227/repo/ML/outputs/robustness_tau/merged.csv


,coid,mdate,tcri,scr,xcdt,company,WorkingCapital_TotalAssets,RetainedEarnings_TotalAssets,CashFlow_TotalDebt,TotalDebt_TotalAssets,CurrentRatio,stock_prefix,WorkingCapital_TotalAssets_miss,RetainedEarnings_TotalAssets_miss,CashFlow_TotalDebt_miss,TotalDebt_TotalAssets_miss,CurrentRatio_miss,產業別,GICS_Category
0,1103,2014-12-01,6,400.0,NaN,1103 嘉泥,0.248834,0.222521,0.031686,0.355758,1.573942,11,0,0,0,0,0,水泥工業,9
1,1103,2015-12-01,6,432.0,NaN,1103 嘉泥,0.288490,0.266805,0.022494,0.396789,2.079358,11,0,0,0,0,0,水泥工業,9
2,1103,2016-12-01,6,371.0,NaN,1103 嘉泥,0.251213,0.276883,0.039433,0.332750,1.751954,11,0,0,0,0,0,水泥工業,9
3,1103,2017-12-01,6,330.0,NaN,1103 嘉泥,0.214906,0.277971,0.043575,0.277508,1.595351,11,0,0,0,0,0,水泥工業,9
4,1103,2018-12-01,6,331.0,NaN,1103 嘉泥,0.222120,0.264199,-0.055883,0.297109,1.501782,11,0,0,0,0,0,水泥工業,9


In [3]:
import os, json, pandas as pd, subprocess

ROOT_OUT  = 'outputs/robustness_tau'
MERGED_PATH = os.path.join(ROOT_OUT, 'merged.csv')
TAUS = [6, 7, 8]  # baseline 原本是 7

results = {}
for tau in TAUS:
    outdir = os.path.join(ROOT_OUT, f'tau{tau}')
    os.makedirs(outdir, exist_ok=True)
    print(f'\n=== Running tau={tau} -> {outdir} ===')
    cmd = [
        'python', 'tcri_baseline_logit.py',
        '--csv', MERGED_PATH,
        '--gics-col', 'GICS_Category',
        '--tau', str(tau),
        '--train-start', '2014-01-01', '--train-end', '2021-12-31',
        '--valid-start', '2022-01-01', '--valid-end', '2022-12-31',
        '--test-start',  '2023-01-01', '--test-end',  '2023-12-31',
        '--outdir', outdir,
    ]
    subprocess.run(cmd, check=True)
    with open(os.path.join(outdir, 'metrics_summary.json'), 'r', encoding='utf-8') as f:
        results[tau] = json.load(f)
print('Done.')



=== Running tau=6 -> outputs/robustness_tau/tau6 ===
[INFO] Selected C=10.0 by PR-AUC on validation (0.8875)
[INFO] Thresholds on validation: t_f1=0.2260, t_at_P>=0.50 = 0.0152


/Users/chieh.1227/repo/ML/.venv/lib/python3.12/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


SHAP plots saved under outputs/plots: shap_summary_baseline_logit.png, shap_bar_baseline_logit.png

=== Test Overall Metrics (threshold = t_f1 on valid) ===
[raw] PR-AUC=0.7488  ROC-AUC=0.6962  F1=0.7378  Recall@P>=0.50=1.0000  Brier=0.2326  ECE=0.1196
[platt] PR-AUC=0.7488  ROC-AUC=0.6962  F1=0.7296  Recall@P>=0.50=1.0000  Brier=0.2352  ECE=0.1407

Top 10 feature weights by |coefficient|:
                          feature  coefficient  abs_coefficient
num__RetainedEarnings_TotalAssets    -1.571057         1.571057
            cat__GICS_Category_10     1.201089         1.201089
             cat__GICS_Category_7    -0.908124         0.908124
          num__CashFlow_TotalDebt    -0.501718         0.501718
       num__TotalDebt_TotalAssets    -0.461384         0.461384
             cat__GICS_Category_5     0.311831         0.311831
             cat__GICS_Category_6    -0.305844         0.305844
  num__WorkingCapital_TotalAssets    -0.278153         0.278153
             cat__GICS_Category

/Users/chieh.1227/repo/ML/.venv/lib/python3.12/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


SHAP plots saved under outputs/plots: shap_summary_baseline_logit.png, shap_bar_baseline_logit.png

=== Test Overall Metrics (threshold = t_f1 on valid) ===
[raw] PR-AUC=0.6542  ROC-AUC=0.8018  F1=0.6130  Recall@P>=0.50=0.7989  Brier=0.1699  ECE=0.1121
[platt] PR-AUC=0.6542  ROC-AUC=0.8018  F1=0.5977  Recall@P>=0.50=0.7989  Brier=0.1575  ECE=0.0633

Top 10 feature weights by |coefficient|:
                          feature  coefficient  abs_coefficient
             cat__GICS_Category_7    -2.960184         2.960184
num__RetainedEarnings_TotalAssets    -1.844111         1.844111
            cat__GICS_Category_10     1.700931         1.700931
          num__CashFlow_TotalDebt    -0.527946         0.527946
             cat__GICS_Category_9    -0.507665         0.507665
  num__WorkingCapital_TotalAssets    -0.437205         0.437205
             cat__GICS_Category_8     0.411531         0.411531
            cat__GICS_Category_11     0.360624         0.360624
             cat__GICS_Category

/Users/chieh.1227/repo/ML/.venv/lib/python3.12/site-packages/sklearn/calibration.py:330: FutureWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


SHAP plots saved under outputs/plots: shap_summary_baseline_logit.png, shap_bar_baseline_logit.png

=== Test Overall Metrics (threshold = t_f1 on valid) ===
[raw] PR-AUC=0.5778  ROC-AUC=0.8903  F1=0.5730  Recall@P>=0.50=0.6842  Brier=0.1139  ECE=0.1661
[platt] PR-AUC=0.5778  ROC-AUC=0.8903  F1=0.4444  Recall@P>=0.50=0.6842  Brier=0.0736  ECE=0.0313

Top 10 feature weights by |coefficient|:
                          feature  coefficient  abs_coefficient
num__RetainedEarnings_TotalAssets    -1.782676         1.782676
            cat__GICS_Category_10     1.290820         1.290820
             cat__GICS_Category_9    -0.925143         0.925143
             cat__GICS_Category_3    -0.698435         0.698435
          num__CashFlow_TotalDebt    -0.578316         0.578316
            cat__GICS_Category_11     0.561583         0.561583
  num__WorkingCapital_TotalAssets    -0.557719         0.557719
                num__CurrentRatio     0.403152         0.403152
             cat__GICS_Category

In [4]:
import pandas as pd

rows = []
for tau, metrics in results.items():
    for name in ['raw', 'platt']:
        m = metrics[name]
        rows.append({
            'tau': tau,
            'calibration': name,
            'pr_auc': m.get('pr_auc'),
            'roc_auc': m.get('roc_auc'),
            'f1': m.get('f1'),
            'precision': m.get('precision'),
            'recall': m.get('recall'),
            'brier': m.get('brier'),
            'ece': m.get('ece'),
        })
df_tau = pd.DataFrame(rows)
df_tau.sort_values(['calibration','tau'])


,tau,calibration,pr_auc,roc_auc,f1,precision,recall,brier,ece
1,6,platt,0.748767,0.696226,0.729560,0.580000,0.983051,0.235200,0.140743
3,7,platt,0.654198,0.801778,0.597701,0.615385,0.581006,0.157525,0.063335
5,8,platt,0.577760,0.890327,0.444444,0.750000,0.315789,0.073633,0.031298
0,6,raw,0.748767,0.696226,0.737819,0.625984,0.898305,0.232563,0.119627
2,7,raw,0.654198,0.801778,0.612987,0.572816,0.659218,0.169858,0.112107
4,8,raw,0.577760,0.890327,0.573034,0.500000,0.671053,0.113915,0.166093
